# Beyond Kubernetes Jobs: live demo

Every cell is plain shell (`%%bash` on the first line): `kubectl`, `cat`, `sleep` and a little `jq`. No Python. Every **▶ NB x.y** chip on a slide is a section here, in the same order.

**Start it:** `./notebook.sh` from the `demo/` directory (it runs JupyterLab through `uvx`, so there's nothing to install).
Then run **NB 0** once before the talk. Creating the cluster takes a few minutes the first time.

| NB | Slide | What you show |
|---|---|---|
| 0 | 9 | The demo cluster: gang scheduling on, 3 workers × 2 fake GPUs |
| 1.1 | 4 | A plain Job |
| 2.1 – 2.5 | 10 – 15 | `hello`: roles, ownership, defaults, events, DNS, coordinator |
| 3.1 | 17 | Success policy |
| 3.2 – 3.5 | 18 – 19 | `resilient`: three failures, three behaviours |
| 4.1 | 22 | `pipeline`: dependsOn |
| 5.1 – 5.2 | 27 – 28 | `train`: Workload, PodGroups, who owns vs. who's scheduled together |
| 5.3 – 5.4 | 29 | Partial placement vs. gang |
| 5.5 | 30 | Elastic scale-down, `minCount` follows |
| 5.6 | 31 | Suspend / resume |
| 6 | – | Clean up |

## ▶ NB 0 · Slide 9: One cluster for every demo

In [ ]:
%%bash
./cluster.sh up

In [ ]:
%%bash
./cluster.sh status

The JobSet controller config, with its feature gates:

In [ ]:
%%bash
kubectl -n jobset-system get configmap jobset-manager-config -o yaml

## ▶ NB 1.1 · Slide 4: A Job runs pods until they finish

In [ ]:
%%bash
cat manifests/00-plain-job.yaml

In [ ]:
%%bash
kubectl apply -f manifests/00-plain-job.yaml
kubectl wait --for=condition=Complete job/hello-batch --timeout=120s
kubectl get pods -l job-name=hello-batch
kubectl logs -l job-name=hello-batch --prefix

In [ ]:
%%bash
kubectl delete -f manifests/00-plain-job.yaml

## ▶ NB 2.1 · Slide 10: One manifest, two roles, DNS, coordinator, success policy

Ask the room to predict: how many Jobs, Pods and Services?

In [ ]:
%%bash
cat manifests/01-hello.yaml

In [ ]:
%%bash
kubectl apply -f manifests/01-hello.yaml

## ▶ NB 2.2 · Slide 12: Apply it: 3 Jobs, 5 Pods

The workers run for about 90 seconds, which leaves time for 2.3 – 2.5 before the success policy kicks in.

In [ ]:
%%bash
sleep 5
kubectl wait --for=condition=Ready pod -l jobset.sigs.k8s.io/jobset-name=hello --timeout=120s
kubectl get jobset hello
kubectl get jobs -l jobset.sigs.k8s.io/jobset-name=hello
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=hello -L jobset.sigs.k8s.io/replicatedjob-name,jobset.sigs.k8s.io/job-index

## ▶ NB 2.3 · Slide 13: What actually exists in the cluster

Who owns what (`ownerReferences`). There's no ReplicatedJob object, only Jobs labelled with it.

In [ ]:
%%bash
kubectl get svc,jobs,pods -o custom-columns='KIND:.kind,NAME:.metadata.name,OWNER:.metadata.ownerReferences[0].name' | grep -E '^KIND|hello'

Same thing as a tree, if you have the krew plugin (`kubectl krew install tree`):

In [ ]:
%%bash
kubectl tree jobset hello || echo "kubectl tree not installed: kubectl krew install tree"

## ▶ NB 2.4 · Slide 14: Watch the background work happen

The stored object: point at the fields you never wrote (`startupPolicy`, `completionMode`, `restartPolicy`, `network`). The webhook added them.

In [ ]:
%%bash
kubectl get jobset hello -o yaml

Who did what: the Job controller created Pods, the scheduler placed them, the kubelet started them.

In [ ]:
%%bash
kubectl get events --sort-by=.lastTimestamp -o custom-columns=FROM:.reportingComponent,REASON:.reason,OBJECT:.involvedObject.name | tail -15

The JobSet controller's view (counts **Jobs**, not Pods):

In [ ]:
%%bash
kubectl get jobset hello -o json | jq -c '.status.replicatedJobsStatus[] | {name, ready, active, succeeded}'

## ▶ NB 2.5 · Slide 15: Stable names, and a coordinator everyone can find

In [ ]:
%%bash
kubectl get svc hello
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=hello -o custom-columns=HOSTNAME:.spec.hostname,SUBDOMAIN:.spec.subdomain

Resolve a worker by name from another pod:

In [ ]:
%%bash
kubectl run dns-test --image=busybox:1.37 --restart=Never -- nslookup hello-workers-1-0.hello
kubectl wait --for=jsonpath='{.status.phase}'=Succeeded pod/dns-test --timeout=60s
kubectl logs dns-test
kubectl delete pod dns-test

Every pod carries the coordinator's address, and the workers used it:

In [ ]:
%%bash
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=hello -L jobset.sigs.k8s.io/coordinator
kubectl logs -l jobset.sigs.k8s.io/jobset-name=hello,jobset.sigs.k8s.io/replicatedjob-name=workers --prefix

## ▶ NB 3.1 · Slide 17: Done when the real work is done

The leader would sleep for an hour. The success policy completes the JobSet when `workers` finish and tears the leader down.

In [ ]:
%%bash
kubectl wait --for=condition=Completed jobset/hello --timeout=240s
kubectl get jobset hello
kubectl get jobs -l jobset.sigs.k8s.io/jobset-name=hello

In [ ]:
%%bash
kubectl delete jobset hello

## ▶ NB 3.2 · Slides 18 – 19: One failure policy, three behaviours

Every pod waits for a trigger file: `/tmp/crash` exits 1 (a bug), `/tmp/preempt` exits 42 (simulated preemption, matched by the Job's `podFailurePolicy`).

In [ ]:
%%bash
cat manifests/02-resilient.yaml

In [ ]:
%%bash
kubectl apply -f manifests/02-resilient.yaml
sleep 5
kubectl wait --for=condition=Ready pod -l jobset.sigs.k8s.io/jobset-name=resilient --timeout=120s
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=resilient

## ▶ NB 3.3 · ① A worker crashes → `RestartJob`

Only `workers-1` is recreated. Check the AGE column.

In [ ]:
%%bash
POD=$(kubectl get pods -l jobset.sigs.k8s.io/jobset-name=resilient,jobset.sigs.k8s.io/replicatedjob-name=workers,jobset.sigs.k8s.io/job-index=1 -o name)
kubectl exec $POD -- touch /tmp/crash
kubectl wait --for=delete $POD --timeout=90s

In [ ]:
%%bash
sleep 5
kubectl wait --for=condition=Ready pod -l jobset.sigs.k8s.io/jobset-name=resilient --timeout=120s
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=resilient
kubectl get jobset resilient -o json | jq -c '.status | {restarts, restartsCountTowardsMax}'

## ▶ NB 3.4 · ② The leader is "preempted" → free `RestartJobSet`

Every pod is new, `restarts` goes up, and `restartsCountTowardsMax` does not.

In [ ]:
%%bash
POD=$(kubectl get pods -l jobset.sigs.k8s.io/jobset-name=resilient,jobset.sigs.k8s.io/replicatedjob-name=leader -o name)
kubectl exec $POD -- touch /tmp/preempt
kubectl wait --for=delete $POD --timeout=90s

In [ ]:
%%bash
sleep 5
kubectl wait --for=condition=Ready pod -l jobset.sigs.k8s.io/jobset-name=resilient --timeout=120s
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=resilient
kubectl get jobset resilient -o json | jq -c '.status | {restarts, restartsCountTowardsMax}'

## ▶ NB 3.5 · ③ The leader has a bug → `RestartJobSet`, spends the budget

In [ ]:
%%bash
POD=$(kubectl get pods -l jobset.sigs.k8s.io/jobset-name=resilient,jobset.sigs.k8s.io/replicatedjob-name=leader -o name)
kubectl exec $POD -- touch /tmp/crash
kubectl wait --for=delete $POD --timeout=90s

In [ ]:
%%bash
sleep 5
kubectl wait --for=condition=Ready pod -l jobset.sigs.k8s.io/jobset-name=resilient --timeout=120s
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=resilient
kubectl get jobset resilient -o json | jq -c '.status | {restarts, restartsCountTowardsMax}'
kubectl describe jobset resilient | sed -n '/Events:/,$p'

*Optional:* run the two NB 3.5 cells 3 more times to spend the `maxRestarts: 3` budget, then:

In [ ]:
%%bash
kubectl get jobset resilient
kubectl get jobset resilient -o jsonpath='{range .status.conditions[*]}{.type}={.status} {.reason}{"\n"}{end}'

In [ ]:
%%bash
kubectl delete jobset resilient

## ▶ NB 4.1 · Slide 22: Watch the dependency graph unfold

`initializer` must **Complete** → `launcher` must be **Ready** → then the `worker` Job is created. Run the three cells a few seconds apart.

In [ ]:
%%bash
cat manifests/03-pipeline.yaml

In [ ]:
%%bash
kubectl apply -f manifests/03-pipeline.yaml
sleep 5
kubectl get jobs -l jobset.sigs.k8s.io/jobset-name=pipeline

In [ ]:
%%bash
sleep 15
kubectl get jobs -l jobset.sigs.k8s.io/jobset-name=pipeline

In [ ]:
%%bash
sleep 15
kubectl get jobs -l jobset.sigs.k8s.io/jobset-name=pipeline

In [ ]:
%%bash
kubectl delete jobset pipeline

## ▶ NB 5.1 · Slide 27: Turn it on: one gang per role

`train` holds 4 of the 6 GPUs and keeps running for the rest of section 5.

In [ ]:
%%bash
cat manifests/04-train.yaml

In [ ]:
%%bash
kubectl apply -f manifests/04-train.yaml
sleep 5
kubectl wait --for=condition=Ready pod -l jobset.sigs.k8s.io/jobset-name=train --timeout=120s
kubectl get workloads.scheduling.k8s.io,podgroups -l jobset.sigs.k8s.io/jobset-name=train

In [ ]:
%%bash
kubectl get podgroups -l jobset.sigs.k8s.io/jobset-name=train -o custom-columns=NAME:.metadata.name,TEMPLATE:.spec.workloadRef.templateName,MIN:.spec.schedulingPolicy.gang.minCount

The Workload (the blueprint) and the PodGroups (the runtime instances), as YAML:

In [ ]:
%%bash
kubectl get workloads.scheduling.k8s.io -l jobset.sigs.k8s.io/jobset-name=train -o yaml

In [ ]:
%%bash
kubectl get podgroups -l jobset.sigs.k8s.io/jobset-name=train -o yaml

How they're wired: the Job's annotation, and each pod's `schedulingGroup`:

In [ ]:
%%bash
kubectl get job train-workers-1 -o jsonpath='{.metadata.annotations.scheduling\.k8s\.io/group-template-name}{"\n"}'
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=train -o custom-columns=POD:.metadata.name,PODGROUP:.spec.schedulingGroup.podGroupName,NODE:.spec.nodeName

## ▶ NB 5.2 · Slide 28: Who owns it vs. who gets scheduled together

Ownership: everything hangs off the JobSet.

In [ ]:
%%bash
kubectl get workloads.scheduling.k8s.io,podgroups,svc,jobs,pods -o custom-columns='KIND:.kind,NAME:.metadata.name,OWNER:.metadata.ownerReferences[0].name' | grep -E '^KIND|train'

Scheduling: pods grouped by PodGroup. The `workers` gang spans **two Jobs**.

In [ ]:
%%bash
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=train --sort-by=.spec.schedulingGroup.podGroupName -o custom-columns='PODGROUP:.spec.schedulingGroup.podGroupName,JOB:.metadata.labels.batch\.kubernetes\.io/job-name,POD:.metadata.name,STATUS:.status.phase'

## ▶ NB 5.3 · Slide 29 (left): Without gang: partial placement

Only 2 GPUs are free. The same workload without `spec.scheduling` grabs them anyway: 2 Running, 2 Pending.

In [ ]:
%%bash
cat manifests/05-big-no-gang.yaml

In [ ]:
%%bash
kubectl apply -f manifests/05-big-no-gang.yaml
sleep 20
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=big-no-gang,jobset.sigs.k8s.io/replicatedjob-name=workers

Delete it so it stops holding the 2 free GPUs:

In [ ]:
%%bash
kubectl delete jobset big-no-gang --wait

## ▶ NB 5.4 · Slide 29 (right): With per-role gangs: all or nothing

All 4 workers stay Pending and hold nothing. The driver's own gang (min 1) still runs.

In [ ]:
%%bash
cat manifests/05-big.yaml

In [ ]:
%%bash
kubectl apply -f manifests/05-big.yaml
sleep 20
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=big

In [ ]:
%%bash
kubectl get podgroups -l jobset.sigs.k8s.io/jobset-name=big -o custom-columns=NAME:.metadata.name,MIN:.spec.schedulingPolicy.gang.minCount,SCHEDULED:.status.conditions[0].status,REASON:.status.conditions[0].reason

## ▶ NB 5.5 · Slide 30: Shrink it in place, and the gang follows

`ElasticJobSet`: patch `parallelism`/`completions` on `workers` (index 1). The PodGroup's `minCount` goes 4 → 2, and both remaining workers bind together.

In [ ]:
%%bash
kubectl patch jobset big --type=json -p '[
  {"op":"replace","path":"/spec/replicatedJobs/1/template/spec/parallelism","value":1},
  {"op":"replace","path":"/spec/replicatedJobs/1/template/spec/completions","value":1}]'

In [ ]:
%%bash
sleep 15
kubectl get podgroups -l jobset.sigs.k8s.io/jobset-name=big -o custom-columns=NAME:.metadata.name,MIN:.spec.schedulingPolicy.gang.minCount
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=big,jobset.sigs.k8s.io/replicatedjob-name=workers

## ▶ NB 5.6 · Slide 31: Suspend releases the gang, resume brings it back

In [ ]:
%%bash
kubectl patch jobset train --type=merge -p '{"spec":{"suspend":true}}'
kubectl wait --for=delete podgroups -l jobset.sigs.k8s.io/jobset-name=train --timeout=60s
kubectl get workloads.scheduling.k8s.io,podgroups,pods -l jobset.sigs.k8s.io/jobset-name=train

In [ ]:
%%bash
kubectl patch jobset train --type=merge -p '{"spec":{"suspend":false}}'
sleep 5
kubectl get podgroups -l jobset.sigs.k8s.io/jobset-name=train
kubectl get pods -l jobset.sigs.k8s.io/jobset-name=train

## ▶ NB 6: Clean up

`reset` deletes every JobSet and Job but keeps the cluster for the next rehearsal. `down` deletes the cluster.

In [ ]:
%%bash
./cluster.sh reset

In [ ]:
%%bash
# ./cluster.sh down